In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import module, secret_keys
from model_list import models
import pandas as pd

hf_api_key             = secret_keys.HF_TOKEN                   #<insert your own huggingface token here>
openai_api_key         = secret_keys.OPENAI_API_KEY_TEAM        #<insert your own openai token here>

In [3]:
data_pub_eval = pd.read_csv('hidden_data/CT-Pub-With-Examples-Corrected-allgpteval.csv')
data_pub_gen = pd.read_csv('hidden_data/CT-Pub-With-Examples-Corrected-allgen.csv')
print(data_pub_eval.shape)
print(data_pub_gen.shape)

(103, 5)
(103, 16)


In [4]:
data_pub_eval.head(2)

,NCTId,gpt4o_zs_gen_matches,gpt4o_ts_gen_matches,llama3_70b_it_zs_gen_matches,llama3_70b_it_ts_gen_matches
0,NCT00000620,"{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ..."
1,NCT00126737,"{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ...","{\n ""matched_features"": [\n [""Age"", ..."


In [5]:
data_pub_gen.head(2)

,NCTId,BriefTitle,EligibilityCriteria,BriefSummary,Conditions,Interventions,PrimaryOutcomes,TrialGroup,API_BaselineMeasures,API_BaselineMeasures_Corrected,Paper_BaselineMeasures,Paper_BaselineMeasures_Corrected,gpt4o_zs_gen,gpt4o_ts_gen,llama3_70b_it_zs_gen,llama3_70b_it_ts_gen
0,NCT00000620,Action to Control Cardiovascular Risk in Diabe...,Inclusion Criteria:\n\n* Diagnosed with type 2...,The purpose of this study is to prevent major ...,"Atherosclerosis, Cardiovascular Diseases, Hype...","Anti-hyperglycemic Agents, Anti-hypertensive A...",First Occurrence of a Major Cardiovascular Eve...,hypertension,"Age, Continuous, Gender, Ethnicity (NIH/OMB), ...","`Age, Continuous`, `Gender`, `Ethnicity (NIH/O...","Age, Female sex, Median duration of diabetes, ...","`Age`, `Female sex`, `Median duration of diabe...","`Age`, `Sex`, `Race/Ethnicity`, `Body Mass Ind...","`Age`, `Female sex`, `Median duration of diabe...","`Age`, `Sex`, `Race`, `Ethnicity`, `Body Mass ...","`Age`, `Female sex`, `Median duration of diabe..."
1,NCT00126737,Home-Based Exercise and Weight Control Program...,Inclusion Criteria:\n\n* Male \& female 50 yea...,The purpose of this study is to determine whet...,"Chronic Diseases, Obesity, Osteoarthritis, Pain,","Weight Control Nutritional Program, Home-based...","WOMAC Function, Physical Scale SF-36v, Mental ...",obesity,"Age, Continuous, Sex: Female, Male, Race/Ethni...","`Age, Continuous`, `Sex: Female, Male`, `Race/...","Age, Duration of OA, Kellgren-Lawrence Classif...","`Age`, `Duration of OA`, `Kellgren-Lawrence Cl...","`Age`, `Gender`, `Body Mass Index (BMI)`, `Kel...","`Age`, `Sex`, `BMI`, `Knee radiographs grade`,...","`Age`, `Gender`, `Body Mass Index (BMI)`, `Kel...","`Age`, `Sex`, `Body Mass Index (BMI)`, `Kellgr..."


## Example Hallucination Calculation Check 

In [6]:

#`Gender` in reference and `Inflammation` in candidate are Negative hallucinations, not reported in matches or remainings 
reference_features = ['`Age`', '`Blood Pressure`', '`Height`', '`Gender`', '`Previous Medication`', '`Race`', '`Ethnicity`']
candidate_features = ['`Age`', '`Systolic Blood Pressure`', '`Diastolic Blood Pressure`', '`Body Mass Index`', '`Race`', "`Inflammation`"]

matched_results =   {
                        "matched_features": [
                            ["`Age`", "`Age`"],
                            ["`bogus1`", "`bogus 2"], ## <-- positive hallucinations in both reference and candidate, but we count only 1 for each matched pair
                            ["`body mass index`", "`Body Mass Index`"], ## <-- positive hallucination 'body mass index' doesn't exist in reference
                            ["`Blood Pressure`", "`Systolic Blood Pressure`"], ## <-- multimatch hallucination in reference (counting one of multiple matches as correct match)
                            ["`Blood Pressure`", "`Diastolic Blood Pressure`"],## <-- multimatch hallucination in reference
                            ["`Race`","`Race`"], ## <-- multimatch hallucination in candidate (counting one of multiple matches as correct match)
                            ["`Ethnicity`", "`Race`"], ## <-- multimatch hallucination in candidate
                            ["`Height`", "`patient height`"] ##<-- positive hallucination 'patient height' doesn't exist in candidate
                        ],
                        "remaining_reference_features": ["`Previous Medication`"],
                        "remaining_candidate_features": []
                    }

module.calculate_hallucination(reference_features, candidate_features, matched_results)

(3, 2, 2, 3)

## Calculate for whole CT-Pub Dataset and save in dataframe

In [7]:
pub_hallucination_results = pd.DataFrame()
pub_hallucination_results['NCTId'] = data_pub_gen['NCTId']
pub_hallucination_results['TrialGroup'] = data_pub_gen['TrialGroup']

In [8]:
import json 
ref_column_name = 'Paper_BaselineMeasures_Corrected'

for index, row_gen in data_pub_gen.iterrows():
    avoid_ids = ['NCT00000620', 'NCT01483560', 'NCT04280783'] #these were used as examples for 3-shot generation
    if row_gen['NCTId'] in avoid_ids:
        continue

    row_eval = data_pub_eval[data_pub_eval['NCTId'] == row_gen['NCTId']]
    if row_eval.empty:
        print(f"Missing NCTId {row_gen['NCTId']} in data_pub_eval")
        continue
    reference_features = module.extract_elements_v2(row_gen[ref_column_name])

    #calculate adjusted precision, recall and f1 for GPT4 zero shot generation
    gzs_candidate = module.extract_elements_v2(row_gen['gpt4o_zs_gen'])
    gzs_matches = json.loads(row_eval['gpt4o_zs_gen_matches'].values[0])
    gzs_hallucination = module.calculate_hallucination(reference_features, gzs_candidate, gzs_matches)
    if 'gpt4o_zs_gen_hal' not in pub_hallucination_results.columns:
        pub_hallucination_results['gpt4o_zs_gen_hal'] = None
    gzs_precision = gzs_hallucination[3]/len(gzs_candidate)
    gzs_recall = gzs_hallucination[3]/len(reference_features)
    gzs_f1 = 2 * (gzs_precision * gzs_recall) / (gzs_precision + gzs_recall) if gzs_precision + gzs_recall > 0 else 0
    pub_hallucination_results.at[index, 'gpt4o_zs_gen_hal'] = (gzs_hallucination[0], gzs_hallucination[1], gzs_hallucination[2], gzs_hallucination[3], gzs_precision, gzs_recall, gzs_f1)

    #calculate adjusted precision, recall and f1 for GPT4 three shot generation
    gts_candidate = module.extract_elements_v2(row_gen['gpt4o_ts_gen'])
    gts_matches = json.loads(row_eval['gpt4o_ts_gen_matches'].values[0])
    gts_hallucination = module.calculate_hallucination(reference_features, gts_candidate, gts_matches)
    if 'gpt4o_ts_gen_hal' not in pub_hallucination_results.columns:
        pub_hallucination_results['gpt4o_ts_gen_hal'] = None
    gts_precision = gts_hallucination[3]/len(gts_candidate)
    gts_recall = gts_hallucination[3]/len(reference_features)
    gts_f1 = 2 * (gts_precision * gts_recall) / (gts_precision + gts_recall) if gts_precision + gts_recall > 0 else 0
    pub_hallucination_results.at[index, 'gpt4o_ts_gen_hal'] = (gts_hallucination[0], gts_hallucination[1], gts_hallucination[2], gts_hallucination[3], gts_precision, gts_recall, gts_f1)

    #calculate adjusted precision, recall and f1 for LLAMA3 zero shot generation
    lzs_candidate = module.extract_elements_v2(row_gen['llama3_70b_it_zs_gen'])
    lzs_matches = json.loads(row_eval['llama3_70b_it_zs_gen_matches'].values[0])
    lzs_hallucination = module.calculate_hallucination(reference_features, lzs_candidate, lzs_matches)
    if 'llama3_70b_it_zs_gen_hal' not in pub_hallucination_results.columns:
        pub_hallucination_results['llama3_70b_it_zs_gen_hal'] = None
    lzs_precision = lzs_hallucination[3]/len(lzs_candidate)
    lzs_recall = lzs_hallucination[3]/len(reference_features)
    lzs_f1 = 2 * (lzs_precision * lzs_recall) / (lzs_precision + lzs_recall) if lzs_precision + lzs_recall > 0 else 0
    pub_hallucination_results.at[index, 'llama3_70b_it_zs_gen_hal'] = (lzs_hallucination[0], lzs_hallucination[1], lzs_hallucination[2], lzs_hallucination[3], lzs_precision, lzs_recall, lzs_f1)

    #calculate adjusted precision, recall and f1 for LLAMA3 three shot generation
    lts_candidate = module.extract_elements_v2(row_gen['llama3_70b_it_ts_gen'])
    lts_matches = json.loads(row_eval['llama3_70b_it_ts_gen_matches'].values[0])
    lts_hallucination = module.calculate_hallucination(reference_features, lts_candidate, lts_matches)
    if 'llama3_70b_it_ts_gen_hal' not in pub_hallucination_results.columns:
        pub_hallucination_results['llama3_70b_it_ts_gen_hal'] = None
    lts_precision = lts_hallucination[3]/len(lts_candidate)
    lts_recall = lts_hallucination[3]/len(reference_features)
    lts_f1 = 2 * (lts_precision * lts_recall) / (lts_precision + lts_recall) if lts_precision + lts_recall > 0 else 0
    pub_hallucination_results.at[index, 'llama3_70b_it_ts_gen_hal'] = (lts_hallucination[0], lts_hallucination[1], lts_hallucination[2], lts_hallucination[3], lts_precision, lts_recall, lts_f1)


In [9]:
pub_hallucination_results

,NCTId,TrialGroup,gpt4o_zs_gen_hal,gpt4o_ts_gen_hal,llama3_70b_it_zs_gen_hal,llama3_70b_it_ts_gen_hal
0,NCT00000620,hypertension,None,None,None,None
1,NCT00126737,obesity,"(0, 0, 0, 5, 0.5555555555555556, 0.41666666666...","(0, 0, 0, 7, 0.35, 0.5833333333333334, 0.4375)","(0, 0, 0, 4, 0.36363636363636365, 0.3333333333...","(0, 0, 0, 5, 0.4166666666666667, 0.41666666666..."
2,NCT00283686,hypertension,"(15, 0, 0, 3, 0.14285714285714285, 0.078947368...","(2, 0, 1, 5, 0.2631578947368421, 0.13157894736...","(0, 1, 0, 6, 0.375, 0.15789473684210525, 0.222...","(0, 0, 1, 4, 0.2, 0.10526315789473684, 0.13793..."
3,NCT00329030,cancer,"(0, 0, 0, 4, 0.17391304347826086, 0.5, 0.25806...","(0, 0, 0, 5, 0.3125, 0.625, 0.4166666666666667)","(0, 0, 0, 6, 0.3, 0.75, 0.4285714285714285)","(0, 0, 0, 5, 0.3125, 0.625, 0.4166666666666667)"
4,NCT00360334,diabetes,"(0, 0, 0, 7, 0.4666666666666667, 0.77777777777...","(0, 0, 0, 9, 0.47368421052631576, 1.0, 0.64285...","(0, 0, 0, 8, 0.5333333333333333, 0.88888888888...","(0, 0, 0, 7, 0.35, 0.7777777777777778, 0.48275..."
...,...,...,...,...,...,...
98,NCT03890588,chronic kidney disease,"(0, 0, 1, 8, 0.42105263157894735, 0.8, 0.55172...","(0, 0, 0, 10, 0.5555555555555556, 1.0, 0.71428...","(0, 0, 1, 9, 0.5294117647058824, 0.9, 0.666666...","(0, 0, 0, 10, 0.625, 1.0, 0.7692307692307693)"
99,NCT03987919,diabetes,"(0, 0, 0, 12, 0.6666666666666666, 0.8571428571...","(0, 0, 0, 8, 0.47058823529411764, 0.5714285714...","(1, 0, 0, 9, 0.6428571428571429, 0.64285714285...","(0, 0, 0, 10, 0.5, 0.7142857142857143, 0.58823..."
100,NCT04280783,hypertension,None,None,None,None
101,NCT04392375,hypertension,"(2, 1, 0, 10, 0.625, 0.37037037037037035, 0.46...","(0, 0, 1, 9, 0.5294117647058824, 0.33333333333...","(0, 0, 0, 11, 0.6111111111111112, 0.4074074074...","(2, 0, 0, 8, 0.5, 0.2962962962962963, 0.372093..."


# Workshop Hallucination Record Generation

In [10]:
import pandas as pd

def transform_to_long_format(pub_hallucination_results):
    """
    Transforms the given DataFrame containing model results into a long format.

    Parameters:
    pub_hallucination_results (pd.DataFrame): The DataFrame with columns for model results.

    Returns:
    pd.DataFrame: Transformed DataFrame in the long format with detailed metrics.
    """
    # Melt and extract tuples into a long format
    long_format = pub_hallucination_results.melt(
        id_vars=['NCTId', 'TrialGroup'],
        value_vars=['gpt4o_zs_gen_hal', 'gpt4o_ts_gen_hal', 'llama3_70b_it_zs_gen_hal', 'llama3_70b_it_ts_gen_hal'],
        var_name='Generation Model',
        value_name='Metrics'
    )

    # Ensure that all entries in the 'Metrics' column are tuples of length 7
    long_format['Metrics'] = long_format['Metrics'].apply(lambda x: x if isinstance(x, tuple) and len(x) == 7 else (None,) * 7)

    # Expand the tuples into their respective columns
    long_format[['Positive Hallucination', 'Negative Hallucination', 'Multi-match Hallucination', 
                 'Correct Matches', 'Precision', 'Recall', 'F1']] = pd.DataFrame(
        long_format['Metrics'].tolist(), index=long_format.index
    )

    # Drop the original 'Metrics' column as it's no longer needed
    long_format.drop(columns=['Metrics'], inplace=True)

    return long_format


In [11]:
long_data = transform_to_long_format(pub_hallucination_results)
long_data.to_csv('workshop_results/CT_Pub_hallucination_results.csv', index=False)
long_data.head(5)

,NCTId,TrialGroup,Generation Model,Positive Hallucination,Negative Hallucination,Multi-match Hallucination,Correct Matches,Precision,Recall,F1
0,NCT00000620,hypertension,gpt4o_zs_gen_hal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NCT00126737,obesity,gpt4o_zs_gen_hal,0.0,0.0,0.0,5.0,0.555556,0.416667,0.476190
2,NCT00283686,hypertension,gpt4o_zs_gen_hal,15.0,0.0,0.0,3.0,0.142857,0.078947,0.101695
3,NCT00329030,cancer,gpt4o_zs_gen_hal,0.0,0.0,0.0,4.0,0.173913,0.500000,0.258065
4,NCT00360334,diabetes,gpt4o_zs_gen_hal,0.0,0.0,0.0,7.0,0.466667,0.777778,0.583333


# Score calculation (Avg)

In [12]:
#calculate average precision, recall and f1 for each model
#average over all examples, save in separate dataframe
adjusted_scores = pd.DataFrame()
adjusted_scores["Metric"] = ["Adjusted Precision", "Adjusted Recall", "Adjusted F1"]
models = ['gpt4o_zs_gen_hal', 'gpt4o_ts_gen_hal', 'llama3_70b_it_zs_gen_hal', 'llama3_70b_it_ts_gen_hal']

#remove None values from the dataframe
pub_hallucination_results = pub_hallucination_results.dropna()
print(pub_hallucination_results.shape)

for model in models:
    adjusted_scores[model] = [pub_hallucination_results[model].apply(lambda x: x[4]).mean(), #precision mean
                              pub_hallucination_results[model].apply(lambda x: x[5]).mean(), #recall mean 
                              pub_hallucination_results[model].apply(lambda x: x[6]).mean()] #f1 mean 

adjusted_scores

(100, 6)


,Metric,gpt4o_zs_gen_hal,gpt4o_ts_gen_hal,llama3_70b_it_zs_gen_hal,llama3_70b_it_ts_gen_hal
0,Adjusted Precision,0.363237,0.385463,0.425911,0.420470
1,Adjusted Recall,0.496701,0.555304,0.549683,0.570561
2,Adjusted F1,0.398625,0.434106,0.459405,0.460807


# Grouped Score Calculation by TrialGroup

In [13]:
# Group by TrialGroup and calculate mean precision, recall, and F1 scores for each model
grouped_scores = pub_hallucination_results.groupby('TrialGroup').apply(
    lambda x: pd.Series({
        'gpt4o_zero_shot_precision': x['gpt4o_zs_gen_hal'].apply(lambda y: y[4]).mean(),
        'gpt4o_zero_shot_recall': x['gpt4o_zs_gen_hal'].apply(lambda y: y[5]).mean(),
        'gpt4o_zero_shot_f1': x['gpt4o_zs_gen_hal'].apply(lambda y: y[6]).mean(),
        'gpt4o_three_shot_precision': x['gpt4o_ts_gen_hal'].apply(lambda y: y[4]).mean(),
        'gpt4o_three_shot_recall': x['gpt4o_ts_gen_hal'].apply(lambda y: y[5]).mean(),
        'gpt4o_three_shot_f1': x['gpt4o_ts_gen_hal'].apply(lambda y: y[6]).mean(),
        'llama3_zero_shot_precision': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[4]).mean(),
        'llama3_zero_shot_recall': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[5]).mean(),
        'llama3_zero_shot_f1': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[6]).mean(),
        'llama3_three_shot_precision': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[4]).mean(),
        'llama3_three_shot_recall': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[5]).mean(),
        'llama3_three_shot_f1': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[6]).mean(),
    })
).reset_index()

grouped_scores.T

/var/folders/bg/dcwgngc506s6ppbk4kcfrwgr0000gn/T/ipykernel_37856/2790406021.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_scores = pub_hallucination_results.groupby('TrialGroup').apply(


,0,1,2,3,4
TrialGroup,cancer,chronic kidney disease,diabetes,hypertension,obesity
gpt4o_zero_shot_precision,0.243773,0.401743,0.404663,0.385296,0.335513
gpt4o_zero_shot_recall,0.500588,0.514051,0.544461,0.458903,0.415081
gpt4o_zero_shot_f1,0.311876,0.435379,0.447262,0.388298,0.355146
gpt4o_three_shot_precision,0.289311,0.419275,0.411572,0.435453,0.348918
gpt4o_three_shot_recall,0.540003,0.575773,0.587339,0.525673,0.510973
gpt4o_three_shot_f1,0.36175,0.472256,0.463661,0.44149,0.398705
llama3_zero_shot_precision,0.346685,0.473699,0.472211,0.47498,0.322926
llama3_zero_shot_recall,0.586641,0.559617,0.57179,0.551149,0.464003
llama3_zero_shot_f1,0.421488,0.492469,0.499719,0.484176,0.364627
